In [ ]:
# =========================
# TASK 1 - DATA IMPORT & SETUP
# =========================
import pandas as pd
# -------------------------
# STEP 1: Import CSV File
# -------------------------
url="https://raw.githubusercontent.com/jeromivj-git/Social-Media-Engagement-Analytics-Using-Python/refs/heads/main/social_media_engagement_5000.csv"
df=pd.read_csv(url)
print(df.head())
# -------------------------
# STEP 2: Check Data Types
# -------------------------
print("--- Current Datatypes ---")
print("-" * 30)
print(df.dtypes)
print("-" * 35)

#Numeric/Non-Numeric Datatype
numeric_df=df.select_dtypes(include="number")
non_numeric_df=df.select_dtypes(exclude="number")
print("Numeric Columns")
print("-"*25)
print(numeric_df)
print("-"*25)
print("Non-Numeric Columns")
print("-"*25)
print(non_numeric_df)

# -------------------------
# STEP 3: Convert Data Types
# -------------------------
df['posted_at']=pd.to_datetime(df['posted_at'])
print(df.dtypes)



Task 2 — Data Cleaning
 Cleaning Missing Data
Detect missing values (isnull(), isna())
Handle using: dropna(), fillna(), median/mode, forward/backward-fill
Duplicate Handling- Identify & remove duplicates
Data Formatting
Fix incorrect data types
Standardize categories (e.g., gender labels)
Correct unrealistic values in likes, comments, shares
 Feature Cleaning
Extract hashtag count
Clean sentiment labels


In [ ]:
# ---------------------------
# STEP 1: Detect Null Values
# ---------------------------
print(df.isna().sum())
print("="*80)
# ---------------------------
# STEP 2: Handle Null Values
# ---------------------------
print("After Cleaning:\n")
for col in df.columns:
  if df[col].dtype in['int64','float64']:
    df[col]=df[col].fillna(df[col].median())
  elif df[col].dtype == 'object':
    df[col]=df[col].fillna(df[col].mode()[0])
print(df.isnull().sum())




In [ ]:
# -------------------------------
# STEP 3: Handle Duplicate Values
# -------------------------------
print(df.duplicated().sum())


In [ ]:
# -------------------------------
# STEP 4: ■ Standardize categories (e.g., gender labels)
# -------------------------------
for col in df.select_dtypes(include='object'):
  df[col]=df[col].str.strip().str.title()
  print(f"\n--- {col} ---")
  print(df[col].unique())


■ Correct unrealistic values in likes, comments, shares
Common unrealistic values:

negative numbers
extremely large values
nulls in important metrics
text values inside numeric columns

In [ ]:
#step1: — Check summary statistics
print(df[['likes','comments','shares']].describe())
#step2:— Find negative values This code finds and displays all rows where the 'likes','comments','shares values are negative.
print(df[df['likes']<0])
print(df[df['comments']<0])
print(df[df['shares']<0])
cols = ['likes', 'comments', 'shares']

for col in cols:
    df.loc[df[col] < 0, col] = pd.NA



In [ ]:
for col in ['likes', 'comments', 'shares']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"\n{col} outliers:")
    print(outliers[[col]])

●  Feature Cleaning
■ Extract hashtag count
■ Clean sentiment labels

In [ ]:
#df['hashtag_count'] = df['hashtags'].str.count('#')
#print(df['hashtag_count'])
import re

all_tags = []

for text in df['hashtags']:
    tags = re.findall(r"#\w+", str(text))
    all_tags.extend(tags)

from collections import Counter

hashtag_count = Counter(all_tags)

#print(hashtag_count)
for tags, count in hashtag_count.items():
    print(tags, ":", count)
df['sentiment']=df['sentiment'].str.strip().str.title()

print(df['sentiment'])

❖ Task 3 — Data Exploration using Pandas
Perform the following:
● View dataset structure using head(), tail(), shape, and columns.
● Check data types and info with info() and dtypes.
● Generate summary statistics using describe().
● Analyze categorical distributions using value_counts(), unique(), and
nunique().
● Create a correlation matrix for numeric fields.
● Use groupby() to summarize metrics (e.g., avg likes by post type, impressions
by country).

In [ ]:
print("Dataset with Head Function:\n","-"*80,"\n",df.head())
print("-"*75)
print("Dataset with Tail Function:\n","-"*80,"\n",df.tail())
print("-"*75)
print("Shape of Dataset is:\n","-"*80,"\n",df.shape)
print("-"*75)
print("Total columns:\n","-"*80,"\n",df.columns)
print("-"*75)
print("Data Types:\n","-"*80,"\n",df.dtypes)
print("-"*75)
print("DataSet Information:\n","-"*80,"\n",df.info)
print("Summary Statistics:\n","-"*80,"\n",df.describe())

● Analyze categorical distributions using value_counts(), unique(), and nunique(). ● Create a correlation matrix for numeric fields. ● Use groupby() to summarize metrics (e.g., avg likes by post type, impressions by country).

In [ ]:
print(df['gender'].value_counts())
print("-"*100)
print(df['country'].value_counts())
print("-"*100)
print(df['gender'].unique())
print("-"*100)
print("Types of genders: ",df['gender'].nunique())
print("-"*100)
print("Number Of Unique Countries:",df['country'].nunique())
print("-"*100)
print("Correlation Matrix:\n")
print(df.corr(numeric_only=True))
print("-"*100)
avg_likes=df.groupby('post_type')['likes'].mean()
print(avg_likes)
print("-"*100)
country_impressions = df.groupby('country')['impression_count'].sum()
print(country_impressions)
print("-"*100)
summary = df.groupby('post_category')[['likes', 'comments', 'shares']].mean()

print(summary)

In [ ]:
df['post_type'].unique()

❖ Task 4 — Data Wrangling  
● Use merge, concat, or join if combining DataFrames.
● Create new fields such as engagement_score, log-transformed metrics (optional),
and hashtag count.
● Perform groupby summaries by post_type, country, and sentiment.

In [ ]:
import pandas as pd
import numpy as np
import re
# ======================================
# Create New Calculated Fields
# ======================================

# Engagement Score
df['engagement_score'] = (
    df['likes'] +
    df['comments'] +
    df['shares']
)

# Log-transformed metrics (optional)
df['log_likes'] = np.log1p(df['likes'])
df['log_comments'] = np.log1p(df['comments'])
df['log_shares'] = np.log1p(df['shares'])

# ======================================
# Hashtag Count
# ======================================

def count_hashtags(text):
    tags = re.findall(r"#\w+", str(text))
    return len(tags)

df['hashtag_count'] = df['hashtags'].apply(count_hashtags)

# View new columns
print(df[['engagement_score',
          'log_likes',
          'log_comments',
          'log_shares',
          'hashtag_count']].head())

# ======================================
# GroupBy Summaries
# ======================================

# 1. Summary by post_type
post_type_summary = df.groupby('post_type')[
    ['likes', 'comments', 'shares', 'engagement_score']
].mean()

print("\nAverage Metrics by Post Type")
print(post_type_summary)

# ======================================

# 2. Summary by country
country_summary = df.groupby('country')[
    ['impression_count', 'engagement_score']
].sum()

print("\nTotal Impressions and Engagement by Country")
print(country_summary)

# ======================================

# 3. Summary by sentiment
sentiment_summary = df.groupby('sentiment')[
    ['likes', 'comments', 'shares', 'engagement_score']
].agg(['mean', 'max', 'min'])

print("\nSentiment Analysis Summary")
print(sentiment_summary)

# ======================================
# Additional Summary
# ======================================

# Average watch time by sentiment
watchtime_summary = df.groupby('sentiment')['watch_time_sec'].mean()

print("\nAverage Watch Time by Sentiment")
print(watchtime_summary)

Task 5 — Statistical Analysis  
Compute descriptive stats for “likes, comments, shares, watch_time,
engagement_rate, followers” columns:
● Mean, median, mode
● Standard deviation, variance
● Percentiles
● (Optional) Skewness and kurtosis

In [ ]:

stat_column=['likes',
    'comments',
    'shares',
    'watch_time_sec',
    'engagement_rate',
    'follower_count']
Mean_val=df[stat_column].mean()
print("Mean\n","="*20,"\n")
print(Mean_val)
Median_val=df[stat_column].median()
print("Median\n","="*20,"\n")
print(Median_val)
Mode_val=df[stat_column].mode()
print("Mode\n","="*20,"\n")
print(Mode_val)
STD_val=df[stat_column].std()
print("StandardDeviation\n","="*20,"\n")
print(STD_val)
print("Variance\n","="*20,"\n")
print(df[stat_column].var())
print("Percentile\n","="*20,"\n")
# Percentiles
print(df[stat_column].quantile([0.25, 0.50, 0.75]))
print("Skewness\n","="*20,"\n")
# Skewness
print(df[stat_column].skew())
print("Kurtosis\n","="*20,"\n")
# Kurtosis
print(df[stat_column].kurt())

#OR
print("Descriptive Statistics:\n")
df[stat_column].describe()

❖ Task 6 — Data Visualization (Min. 8 Plots Required)
● Matplotlib
○ Scatter: likes vs impressions
○ Line: daily engagement trend
○ Bar: posts by category
○ Pie: gender distribution
○ Histogram: age
○ Box: engagement rate
● Seaborn
○ Count plot: post type
○ Bar plot: avg likes by category
○ Violin: followers vs sentiment
○ Pair plot: numeric features
○ Heatmap: correlation matrix
○ Swarm plot: engagement vs device
● Plotly (Interactive)
○ Interactive line chart/bar chart/bubble/scatter chart
❖ Final Insights should include the following analysis
● Content Performance
○ Which post types have the highest engagement?
○ Best-performing content category?
○ Which countries have the highest average engagement rate?  
● User Trends
○ How age affects engagement
○ Performance difference for verified accounts
● Behavioral Insights
○ Best time of day for impressions
○ Device type impact on watch time
● Sentiment Analysis
○ Which sentiment performs best
○ Behavior of negative/neutral sentiment posts

● Matplotlib ○ Scatter: likes vs impressions ○ Line: daily engagement trend ○ Bar: posts by category ○ Pie: gender distribution ○ Histogram: age ○ Box: engagement rate

In [ ]:
#Scatter: likes vs impressions
import matplotlib.pyplot as plt
import pandas as pd
plt.figure(figsize=(8,5))
plt.scatter(df['likes'],df['impression_count'])
plt.title("Likes vs Impressions")
plt.xlabel("Likes")
plt.ylabel("Impressions")
plt.show()

In [ ]:
# Line: daily engagement trend
df['posted_at'] = pd.to_datetime(df['posted_at'])
daily_engagement = df.groupby(df['posted_at'].dt.date)[
    'engagement_score'
].mean()

plt.figure(figsize=(10,5))

plt.plot(daily_engagement.index,
         daily_engagement.values,
         color='green',
         marker='o')

plt.title("Daily Engagement Trend")
plt.xlabel("Date")
plt.ylabel("Average Engagement")

plt.xticks(rotation=45)

plt.show()

In [ ]:
#Bar: posts by category
category_counts = df['post_category'].value_counts()

plt.figure(figsize=(8,5))

plt.bar(category_counts.index,
        category_counts.values,
        color='orange')

plt.title("Posts by Category")
plt.xlabel("Category")
plt.ylabel("Number of Posts")

plt.xticks(rotation=45)

plt.show()

In [ ]:
#Pie: gender distribution
gender_counts = df['gender'].value_counts()

plt.figure(figsize=(6,6))

plt.pie(gender_counts.values,
        labels=gender_counts.index,
        autopct='%1.1f%%',
        colors=['pink', 'skyblue', 'lightgreen'])

plt.title("Gender Distribution")

plt.show()

In [ ]:

#Histogram: age
plt.figure(figsize=(8,5))

plt.hist(df['age'],
         bins=10,
         color='purple',
         edgecolor='black')

plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

plt.boxplot(df['engagement_rate'])

plt.title("Box Plot of Engagement Rate")
plt.ylabel("Engagement Rate")

plt.show()

In [ ]:
#Box: engagement rate
plt.figure(figsize=(8,5))

plt.hist(df['likes'],
         bins=15,
         color='red',
         edgecolor='black')

plt.title("Likes Distribution")
plt.xlabel("Likes")
plt.ylabel("Frequency")

plt.show()

In [ ]:
country_impressions = df.groupby('country')[
    'impression_count'
].sum()

plt.figure(figsize=(8,5))

plt.bar(country_impressions.index,
        country_impressions.values,
        color='teal')

plt.title("Country-wise Impressions")
plt.xlabel("Country")
plt.ylabel("Total Impressions")

plt.xticks(rotation=45)

plt.show()

● Seaborn
○ Count plot: post type
○ Bar plot: avg likes by category
○ Violin: followers vs sentiment
○ Pair plot: numeric features
○ Heatmap: correlation matrix
○ Swarm plot: engagement vs device

● Seaborn ○ Count plot: post type

In [ ]:
#Count plot: post type
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

sns.countplot(x='post_type',
              data=df,
              palette='Set2')

plt.title("Count of Post Types")
plt.xlabel("Post Type")
plt.ylabel("Count")

plt.show()

○ Bar plot: avg likes by category

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))

sns.barplot(x='post_category',
            y='likes',
            data=df,
            palette='viridis')

plt.title("Average Likes by Category")
plt.xlabel("Post Category")
plt.ylabel("Average Likes")

plt.xticks(rotation=45)

plt.show()

○ Violin: followers vs sentiment

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))

sns.violinplot(x='sentiment',
               y='follower_count',
               data=df,
               palette='pastel')

plt.title("Followers vs Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Follower Count")

plt.show()

○ Pair plot: numeric features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
numeric_df = df[[
    'likes',
    'comments',
    'shares',
    'watch_time_sec',
    'engagement_rate'
]]

sns.pairplot(numeric_df)

plt.show()

○ Heatmap: correlation matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
numeric_df = df[[
    'likes',
    'comments',
    'shares',
    'watch_time_sec',
    'engagement_rate'
]]
correlation_matrix = numeric_df.corr()
plt.figure(figsize=(8,6))

sns.heatmap(correlation_matrix,
            annot=True,
            cmap='coolwarm')

plt.title("Correlation Matrix Heatmap")

plt.show()

● Plotly (Interactive)
○ Interactive line chart/bar chart/bubble/scatter chart

In [ ]:
import pandas as pd
import plotly.express as px
df['month'] = df['posted_at'].dt.to_period('M').astype(str)

monthly_likes = df.groupby('month')['likes'].mean().reset_index()

fig = px.line(
    monthly_likes,
    x='month',
    y='likes',
    title='Monthly Average Likes'
)

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
avg_likes = df.groupby('post_type')['likes'].mean().reset_index()

fig = px.bar(
    avg_likes,
    x='post_type',
    y='likes',
    title='Average Likes by Post Type'
)

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
fig = px.scatter(
    df,
    x='likes',
    y='comments',
    color='sentiment',
    title='Likes vs Comments'
)

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
fig = px.scatter(
    df,
    x='likes',
    y='shares',
    size='impression_count',
    color='post_type',
    hover_name='country',
    title='Bubble Chart of Social Media Performance'
)

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
post_counts = df['post_type'].value_counts().reset_index()

post_counts.columns = ['post_type', 'count']

fig = px.pie(
    post_counts,
    names='post_type',
    values='count',
    title='Post Type Distribution'
)

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
fig = px.histogram(
    df,
    x='likes',
    nbins=20,
    title='Distribution of Likes'
)

fig.show()

❖ Final Insights should include the following analysis
● Content Performance
○ Which post types have the highest engagement?

In [ ]:
post_type_engagement = (
    df.groupby('post_type')['engagement_rate']
    .mean()
    .sort_values(ascending=False)
)

print(post_type_engagement)

Insights:
------------
`Video` posts achieved the highest engagement rate, showing that audiences interact more with dynamic and visual content.
`Text` posts also performed well, indicating users actively engage with informative or discussion-based content.
`Image` posts received comparatively lower engagement than videos and text content.
Overall, richer content formats like videos appear more effective in attracting audience attention and interaction.


 Best-performing content category?

In [ ]:
category_performance=df.groupby('post_category')['engagement_rate'].mean().sort_values(ascending=False)
print(category_performance)
best_category = category_performance.idxmax()
print("Best_Category:->",best_category)

Insights:
---------------
`Food' posts achieved the highest engagement score, showing strong audience interest and interaction.
`Tech` and `Lifestyle` content also performed well, indicating users prefer informative and relatable posts.
`Music`, `Fitness`, and `Education` showed moderate engagement with stable audience response.
`Travel` and `Fashion` had the lowest scores, suggesting these categories may need improved content strategy or better visual appeal.


Which countries have the highest average engagement rate?  

In [ ]:
country_egt=df.groupby('country')['engagement_rate'].mean().sort_values(ascending=False)
print(country_egt)

Insights:
-----------
`Brazil` has the highest average engagement rate, indicating users from Brazil interact more actively with content compared to other countries.
`Australia`, `France`, and `UAE` also show strong engagement performance with above-average interaction rates.
Countries like `Canada` and `UK` demonstrate moderate engagement levels with consistent audience activity.
`India` and `USA` have the lowest average engagement rates, suggesting comparatively lower user interaction or content responsiveness.


User Trends
○ How age affects engagement

In [ ]:
df['age_group'] = pd.cut(df['age'],
                         bins=[18,25,35,45,55],
                         labels=['18-25','26-35','36-45','46-55'])

age_group_engagement = df.groupby('age_group')['engagement_rate'].mean()

print(age_group_engagement)
import matplotlib.pyplot as plt

age_group_engagement.plot(kind='bar')

plt.title('Average Engagement Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Engagement Rate')

plt.show()

Insights:
----------

Users aged `46–55` showed the highest engagement rate, indicating older users interact more actively with content.
The `18–25` age group also demonstrated strong engagement and consistent platform activity.
Users between `26–45` had comparatively lower engagement rates with similar interaction patterns.
This suggests content may be performing better among mature audiences than middle-age user groups.


 Performance difference for verified accounts

In [ ]:
print(df['is_verified'].value_counts())
verified_performance = (
    df.groupby('is_verified')['engagement_rate']
    .mean()
)

print(verified_performance)
import matplotlib.pyplot as plt

verified_performance.plot(kind='bar')

plt.title('Engagement Rate by Verification Status')
plt.xlabel('Verified Account')
plt.ylabel('Average Engagement Rate')

plt.show()

Insight:
There is not much difference in the performance of verified and non-verified customer.

● Behavioral Insights
○ Best time of day for impressions

In [ ]:
df['month'] = df['posted_at'].dt.month_name()

monthly_impressions = df.groupby('month')['impression_count'].mean()

print(monthly_impressions)
import matplotlib.pyplot as plt

monthly_impressions.plot(kind='bar')

plt.title('monthly_impressions')
plt.xlabel('Month')
plt.ylabel('Average Impressions')

plt.show()

Insights:
`September` recorded the highest average impressions, indicating stronger audience reach during that month.
`February`, `August`, and `October` also showed consistently high impression counts.
`June` and `July` had comparatively lower impressions, suggesting reduced content visibility or audience activity.
Overall, monthly impressions remained relatively stable, with only slight variations across the year.


○ Device type impact on watch time

In [ ]:
df.groupby('device_type')['watch_time_sec'].mean()

Insights:
`Mobile` users recorded the highest average watch time, indicating stronger user engagement on mobile devices.
`Desktop` and `Tablet` users showed very similar watch durations with only minor differences.
The results suggest audiences prefer consuming content on mobile platforms due to convenience and accessibility.
Optimizing content for mobile viewing may help improve overall audience retention and engagement.


● Sentiment Analysis
○ Which sentiment performs best

In [ ]:
sentiment_performance = df.groupby('sentiment')['engagement_rate'].mean().sort_values(ascending=False)

print(sentiment_performance)



Insights:
`Negative` sentiment posts achieved the highest engagement rate, indicating users interact more with emotionally intense or controversial content.
`Neutral` sentiment content also performed well with stable audience interaction levels.
`Positive` sentiment posts showed comparatively lower engagement among the three categories.
This suggests audiences may respond more actively to content that creates stronger emotional reactions.


 Behavior of negative/neutral sentiment posts

In [ ]:
sentiment_behavior = df.groupby('sentiment')[
    ['likes', 'comments', 'shares', 'watch_time_sec', 'engagement_rate']
].mean()

print(sentiment_behavior)

Insights:
`Negative` sentiment posts achieved the highest engagement rate, likes, shares, and watch time, showing that emotionally intense content attracts stronger audience interaction.
`Neutral` posts received the highest average comments, indicating users actively discuss balanced or informational content.
`Positive` sentiment posts showed comparatively lower engagement metrics across most categories.
Overall, audiences appear more responsive to negative and discussion-driven content than purely positive posts.


                              **SUMMARY**
         SOCIAL MEDIA ENGAGEMENT ANALYSIS — SUMMARY REPORT

## Objective

The analysis focused on understanding user engagement patterns, content performance, audience behavior, sentiment impact, and platform trends using social media dataset metrics such as likes, comments, shares, impressions, watch time, and engagement rate.

---

## Key Findings

### Content Performance

* `Video` posts achieved the highest engagement rates, indicating audiences prefer dynamic and interactive content formats.
* `Food` was the best-performing content category, followed by `Tech` and `Lifestyle`, showing strong audience interest in relatable and informative content.
* `Brazil` recorded the highest average engagement rate among all countries, while `India` and `USA` showed comparatively lower engagement levels.

---

## User Trends

* Users aged `46–55` showed the highest engagement rate, suggesting mature audiences interact more actively with content.
* The `18–25` age group also demonstrated strong engagement behavior.
* Verified and non-verified accounts showed only minimal differences in engagement performance.

---

## Behavioral Insights

* `September` generated the highest average impressions, indicating stronger audience reach during that month.
* Mobile users recorded the highest average watch time, highlighting the importance of mobile-friendly content optimization.
* Desktop and tablet users showed similar viewing behavior with slightly lower watch durations.

---

## Sentiment Analysis

* `Negative` sentiment posts produced the highest engagement rates, likes, shares, and watch time.
* `Neutral` posts generated the highest average comments, suggesting users actively engage in discussions around balanced or informational content.
* `Positive` sentiment posts showed comparatively lower overall engagement.

---

## Statistical & Visualization Insights

* Descriptive statistical analysis revealed variability in likes, comments, shares, watch time, and follower counts.
* Correlation heatmaps and pair plots helped identify relationships between engagement metrics.
* Interactive Plotly visualizations improved trend analysis and user behavior interpretation.

---

## Conclusion

The analysis shows that audience engagement is strongly influenced by content format, sentiment, device usage, and demographic behavior. Video-based and emotionally intense content tends to perform best, while mobile users and mature audiences contribute significantly to overall engagement. These insights can help improve content strategy, posting optimization, and audience targeting for better social media performance.



